# Modül 05: Regularization
## Deep Learning Path — Kapsamlı Jupyter Notebook
---
## 📋 İçindekiler
1. [Öğrenme Hedefleri](#1)
2. [Overfitting vs Underfitting](#2)
3. [L1 ve L2 Regularization](#3)
4. [Dropout](#4)
5. [Batch Normalization](#5)
6. [Layer Normalization](#6)
7. [Early Stopping](#7)
8. [Data Augmentation](#8)
9. [TF / PyTorch Kullanımı](#9)
10. [Hiperparametre Deneyleri](#10)
11. [Özet ve Alıştırmalar](#11)


## 1. Öğrenme Hedefleri <a id='1'></a>
- ✅ Bias-variance tradeoff'u matematiksel ve görsel olarak açıklamak
- ✅ L1 ve L2 regularization'ı sıfırdan uygulamak, farklarını kanıtlamak
- ✅ Inverted Dropout'u eğitim ve inference modunda doğru uygulamak
- ✅ Batch Normalization algoritmasının her adımını açıklamak
- ✅ Layer Norm ile Batch Norm'un farkını Transformer bağlamında açıklamak
- ✅ Early stopping'i validation loss takibiyle uygulamak
- ✅ TF ve PyTorch'ta tüm regularization tekniklerini kodlamak


## 2. Overfitting vs Underfitting <a id='2'></a>

### Bias-Variance Tradeoff

$$\text{Toplam Hata} = \text{Bias}^2 + \text{Variance} + \text{Gürültü}$$

| Durum | Bias | Variance | Train Err | Test Err | Çözüm |
|-------|------|----------|-----------|----------|-------|
| Underfitting | Yüksek | Düşük | Yüksek | Yüksek | Daha derin model |
| İyi fit | Orta | Orta | Düşük | Düşük | — |
| Overfitting | Düşük | Yüksek | Düşük | Yüksek | Regularization |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

X = np.linspace(0, 2*np.pi, 30)
y = np.sin(X) + .3*np.random.randn(30)
X_t = np.linspace(0, 2*np.pi, 200)

p1  = np.poly1d(np.polyfit(X, y, 1))
p4  = np.poly1d(np.polyfit(X, y, 4))
p20 = np.poly1d(np.polyfit(X, y, 20))

fig,ax = plt.subplots(figsize=(11,4))
ax.scatter(X, y, c='black', s=30, zorder=5, label='Veri')
ax.plot(X_t, np.sin(X_t), 'g-', lw=2, label='Gerçek sin(x)')
ax.plot(X_t, p1(X_t), 'b--', lw=2, label='Derece 1 — Underfitting')
ax.plot(X_t, p4(X_t), color='orange', lw=2, label='Derece 4 — İyi Fit')
ax.plot(X_t, np.clip(p20(X_t),-3,3), 'r-', lw=2, label='Derece 20 — Overfitting')
ax.set_ylim(-3,3); ax.legend(); ax.set_title('Bias-Variance Tradeoff',fontweight='bold')
plt.tight_layout(); plt.show()

mse = lambda yt,yp: np.mean((yt-yp)**2)
for name,poly,d in [('Derece 1',p1,1),('Derece 4',p4,4),('Derece 20',p20,20)]:
    print(f"{name}: Train={mse(y,poly(X)):.4f}  Test={mse(np.sin(X_t),poly(X_t)):.4f}")


## 3. L1 ve L2 Regularization <a id='3'></a>

$$L_{\text{L2}} = L_{\text{data}} + \frac{\lambda}{2}\sum w_i^2 \quad \Rightarrow \quad w \leftarrow w(1-\eta\lambda) - \eta\nabla w$$

$$L_{\text{L1}} = L_{\text{data}} + \lambda\sum|w_i| \quad \Rightarrow \quad w \leftarrow w - \eta\lambda\,\text{sign}(w) - \eta\nabla w$$

**Temel fark:** L1 bazı ağırlıkları tam sıfıra çeker (seyreklik), L2 küçültür ama sıfırlamaz.


In [ ]:
# L1 vs L2 etkisi
def l2_step(w, lr=.1, lam=.3): return w*(1-lr*lam)
def l1_step(w, lr=.1, lam=.3): return w - lr*lam*np.sign(w)

w0 = np.array([3., -2., .5, -.1, 2.5])
wl2,wl1 = w0.copy(),w0.copy()
for _ in range(20):
    wl2 = l2_step(wl2)
    wl1 = l1_step(wl1)

print(f"{'w₀':>8} {'L2':>10} {'L1':>10}")
print("-"*32)
for a,b,c in zip(w0,wl2,wl1):
    flag = " ← SIFIR!" if abs(c)<.05 else ""
    print(f"{a:>8.3f} {b:>10.4f} {c:>10.4f}{flag}")
print(f"\nL1 sıfır ağırlık: {np.sum(np.abs(wl1)<.05)}/5")
print(f"L2 sıfır ağırlık: {np.sum(np.abs(wl2)<.05)}/5")

# Görsel
fig,ax = plt.subplots(figsize=(8,4))
w_r = np.linspace(-3,3,200)
ax.plot(w_r,.5*w_r**2,'b-',lw=2.5,label='L2: ½w²')
ax.plot(w_r,np.abs(w_r),'r-',lw=2.5,label='L1: |w|')
ax.set_title('L1 vs L2 Ceza Fonksiyonu',fontweight='bold')
ax.set_xlabel('w'); ax.set_ylabel('Ceza'); ax.legend(); ax.grid(True,alpha=.3)
plt.tight_layout(); plt.show()


## 4. Dropout <a id='4'></a>

Eğitim: `mask = Bernoulli(1-p)`, `out = x * mask / (1-p)` (inverted dropout)

Inference: `out = x` — mask YOK, ölçekleme YOK.

In [ ]:
class Dropout:
    def __init__(self,p=.5): self.p=p; self.mask=None; self.training=True
    def forward(self,x):
        if not self.training: return x
        self.mask=(np.random.rand(*x.shape)>self.p).astype(float)
        return x*self.mask/(1-self.p)

np.random.seed(0)
x = np.ones((1,10)); drop=Dropout(.5)
print("Eğitim modu (p=0.5) — 5 pass:")
for i in range(5):
    out=drop.forward(x)
    print(f"  Pass {i+1}: aktif={int(np.sum(out>0))}/10  mean={out.mean():.3f}")

drop.training=False
inf=drop.forward(x)
print(f"\nInference: aktif={int(np.sum(inf>0))}/10  mean={inf.mean():.3f}")
print("✓ Beklenti korunuyor — inverted dropout")


## 5. Batch Normalization <a id='5'></a>

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} \qquad y_i = \gamma\hat{x}_i + \beta$$

- $\gamma, \beta$: öğrenilen parametreler
- $\epsilon = 10^{-5}$: sıfıra bölme önlemi
- Inference'da: $\mu, \sigma^2$ yerine hareketli ortalamalar kullanılır


In [ ]:
class BatchNorm:
    def __init__(self,n,eps=1e-5,mom=.1):
        self.eps,self.mom=eps,mom; self.training=True
        self.gamma=np.ones(n); self.beta=np.zeros(n)
        self.rm=np.zeros(n); self.rv=np.ones(n)
    def forward(self,x):
        if self.training:
            mu=x.mean(0); var=x.var(0)
            xh=(x-mu)/np.sqrt(var+self.eps)
            self.rm=(1-self.mom)*self.rm+self.mom*mu
            self.rv=(1-self.mom)*self.rv+self.mom*var
        else:
            xh=(x-self.rm)/np.sqrt(self.rv+self.eps)
        return self.gamma*xh+self.beta

class LayerNorm:
    def __init__(self,n,eps=1e-5):
        self.eps=eps; self.gamma=np.ones(n); self.beta=np.zeros(n)
    def forward(self,x):
        mu=x.mean(-1,keepdims=True); var=x.var(-1,keepdims=True)
        return self.gamma*(x-mu)/np.sqrt(var+self.eps)+self.beta

x = np.random.randn(32,8)*5+3
bn=BatchNorm(8); ln=LayerNorm(8)
out_bn=bn.forward(x); out_ln=ln.forward(x)

print("Batch Norm  → özellik ortalamaları:", out_bn.mean(0).round(4))
print("Layer Norm  → örnek ortalamaları:  ", out_ln.mean(1)[:4].round(4))
print("\nBatch Norm: sütun bazlı normalize (her özellik)")
print("Layer Norm: satır bazlı normalize (her örnek) → Transformer tercihi")


## 6. Early Stopping <a id='7'></a>

In [ ]:
class EarlyStopping:
    def __init__(self,patience=10,delta=1e-4):
        self.patience=patience; self.delta=delta
        self.best=np.inf; self.counter=0; self.best_ep=0; self.stop=False
    def __call__(self,val_loss,ep):
        if val_loss<self.best-self.delta:
            self.best=val_loss; self.best_ep=ep; self.counter=0
        else:
            self.counter+=1
            if self.counter>=self.patience: self.stop=True
        return self.stop

np.random.seed(5)
es=EarlyStopping(patience=8)
train_h,val_h=[],[]
stopped=None
for ep in range(100):
    tl=1./(ep+1)+.01*np.random.rand()
    vl=1./(ep+1)+.05*np.random.rand()+max(0,(ep-30)*.01)
    train_h.append(tl); val_h.append(vl)
    if es(vl,ep) and stopped is None:
        stopped=ep
        print(f"Early Stop! Epoch {ep}, val={vl:.4f}, en iyi epoch={es.best_ep}")

fig,ax=plt.subplots(figsize=(10,4))
ax.plot(train_h,'b-',lw=2,label='Eğitim Loss')
ax.plot(val_h,'r-',lw=2,label='Validation Loss')
if stopped: ax.axvline(stopped,color='green',lw=2,ls='--',label=f'Stop (ep {stopped})')
ax.set_title('Early Stopping',fontweight='bold')
ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True,alpha=.3)
plt.tight_layout(); plt.show()


## 7. Alıştırmalar <a id='11'></a>

**1.** `Dropout` sınıfına `backward()` metodunu ekle ve gradyan kontrolü yap.

**2.** L2 regularization'ı 2 katmanlı MLP'ye entegre et: `loss += λ/2 * (||W1||² + ||W2||²)`

**3.** Batch Norm'un hareketli ortalama momentumunu 0.01, 0.1, 0.5 ile test et — ne fark oluşuyor?

**4.** `EarlyStopping` sınıfına `restore_best_weights=True` özelliği ekle: en iyi modeli geri yükle.

**5.** Aynı modeli 4 farklı regularization kombinasyonuyla eğit ve test hatasını karşılaştır:
   - Baseline (hiç yok)
   - Sadece L2
   - Sadece Dropout
   - L2 + Dropout + Early Stopping

---
**Sonraki Modül:** Modül 06 — CNN (Konvolüsyon, Pooling, ResNet, Grad-CAM)
